# Signed receipts for every tool your agent runs

A practical recipe for adding a **fail-closed policy gate** and **cryptographically signed, offline-verifiable receipts** to a Claude tool-use loop.

Out of the box, an agent that calls tools leaves no audit trail a third party can independently verify: anything logged is operator-controlled and anything unsigned can be edited after the fact. This recipe wraps each tool call in two steps:

1. **Gate** every tool call against a [Cedar](https://www.cedarpolicy.com/) policy with [`protect-mcp`](https://www.npmjs.com/package/protect-mcp). The gate fails **closed**: on any policy error it denies rather than allows.
2. **Sign** an Ed25519 receipt of each decision. Receipts are [JCS](https://datatracker.ietf.org/doc/html/rfc8785)-canonical and verifiable offline by anyone with the public key, with no vendor in the loop.

We use the Anthropic SDK for a real tool-use loop, gate Claude's tool calls, sign the decisions, then verify the receipts offline and show that tampering breaks verification.

## Setup

Requires Node.js 18+ (for `npx`) and an `ANTHROPIC_API_KEY`.

In [ ]:
%pip install -q anthropic
import shutil

assert shutil.which("node"), "Node.js 18+ is required (provides npx)"
print("node:", shutil.which("node"))

In [ ]:
import json
import os
import pathlib
import subprocess

import anthropic

MODEL = "claude-sonnet-5"  # any current Claude model
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
WORK = pathlib.Path("receipts_demo")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

## Step 1: a Cedar policy and a signing key

The policy permits read-only tools and forbids destructive shell commands. Two idioms matter for real policies:

- The tool input arrives in Cedar at `context.input`, so guard with `has` before reading an attribute; a policy that references a missing attribute errors and does not apply.
- Command matching uses Cedar `like` patterns. Exact set membership (`["rm", ...].contains(context.input.command)`) would never match a real command line such as `rm -rf important_data`.


In [ ]:
# ruff: noqa: S603, S607  (intentional npx calls to the protect-mcp CLI)
policy = """
permit(principal, action == Action::"MCP::Tool::call", resource == Tool::"list_files");
forbid(principal, action == Action::"MCP::Tool::call", resource == Tool::"run_shell")
  when {
    context has input && context.input has command &&
    (context.input.command like "rm *" || context.input.command like "*rm -rf*" ||
     context.input.command like "dd *" || context.input.command like "*mkfs*" ||
     context.input.command like "*shutdown*")
  };
permit(principal, action == Action::"MCP::Tool::call", resource == Tool::"run_shell");
"""
pathlib.Path("cedar").mkdir(exist_ok=True)
pathlib.Path("cedar/policy.cedar").write_text(policy)

subprocess.run(["npx", "-y", "protect-mcp@0.9.6", "init"], check=False)
print("key generated:", pathlib.Path("keys/gateway.json").exists())

## Step 2: gate and sign helpers

`protect-mcp evaluate` exits `2` to deny and `0` to allow (a missing policy denies). `protect-mcp sign` appends an Ed25519 receipt; with no key it records an honest unsigned line rather than failing.

In [ ]:
# ruff: noqa: S603, S607  (intentional npx calls to the protect-mcp CLI)
def gate(tool: str, tool_input: dict) -> bool:
    """Return True if the policy allows this call, False if it denies. Fails closed."""
    r = subprocess.run(
        ["npx", "-y", "protect-mcp@0.9.6", "evaluate",
         "--cedar", "cedar", "--tool", tool, "--input", json.dumps(tool_input)],
        capture_output=True, text=True)
    return r.returncode == 0  # 0 = allow, 2 = deny

def sign(tool: str) -> None:
    subprocess.run(
        ["npx", "-y", "protect-mcp@0.9.6", "sign",
         "--tool", tool, "--receipts", "receipts", "--key", "keys/gateway.json"],
        capture_output=True, text=True)

## Step 3: the tool-use loop

We give Claude two tools, `list_files` and `run_shell`. Before any tool runs, we gate it. Allowed calls execute and get a signed receipt; denied calls return a denial to Claude instead of running.

In [ ]:
TOOLS = [
    {"name": "list_files", "description": "List files in the current directory.",
     "input_schema": {"type": "object", "properties": {}}},
    {"name": "run_shell", "description": "Run a shell command.",
     "input_schema": {"type": "object",
                      "properties": {"command": {"type": "string"}},
                      "required": ["command"]}},
]

def execute(tool: str, tool_input: dict) -> str:
    if tool == "list_files":
        return "\n".join(os.listdir("."))
    if tool == "run_shell":
        return f"(would run: {tool_input.get('command')})"
    return "unknown tool"

def run_agent(prompt: str) -> None:
    print(f"USER: {prompt}")
    messages = [{"role": "user", "content": prompt}]
    while True:
        resp = client.messages.create(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            text = "".join(b.text for b in resp.content if b.type == "text")
            print("CLAUDE:", text.strip())
            return
        results = []
        for tu in tool_uses:
            if gate(tu.name, tu.input):
                out = execute(tu.name, tu.input)
                sign(tu.name)
                print(f"  ALLOW {tu.name} {tu.input}  (receipt signed)")
            else:
                out = "DENIED by policy. This action was blocked before it ran."
                print(f"  DENY  {tu.name} {tu.input}  (fail-closed gate)")
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": out})
        messages.append({"role": "user", "content": results})

## Step 4: watch it allow a safe call and block a dangerous one

In [ ]:
run_agent("List the files in the current directory.")
print("-" * 60)
run_agent("Run the shell command: rm -rf important_data")

## Step 5: verify the receipts offline

The receipts verify offline: no network, no vendor in the loop. Receipts deliberately do not embed their own public key (a record that carries its own trust anchor proves little), so verification checks each signature against the operator's key, which in production you would publish or pin.


In [ ]:
# ruff: noqa: S603, S607  (intentional npx calls to the offline verifier)
PUBKEY = json.loads(pathlib.Path("keys/gateway.json").read_text())["publicKey"]
subprocess.run(["npx", "-y", "@veritasacta/verify@0.9.2",
                "--replay-chain", "receipts/receipts.jsonl", "--key", PUBKEY])

Now tamper with a receipt and verify again. The Ed25519 signature no longer matches the canonical bytes, so verification fails.

In [ ]:
# ruff: noqa: S603, S607  (intentional npx calls to the offline verifier)
lines = pathlib.Path("receipts/receipts.jsonl").read_text().splitlines()
if lines:
    rec = json.loads(lines[0])
    rec["payload"]["tool"] = "tampered"  # the signed facts live in payload
    lines[0] = json.dumps(rec)
    pathlib.Path("receipts/receipts.jsonl").write_text("\n".join(lines) + "\n")
r = subprocess.run(["npx", "-y", "@veritasacta/verify@0.9.2",
                    "--replay-chain", "receipts/receipts.jsonl", "--key", PUBKEY])
print("exit code:", r.returncode, "(non-zero = tamper detected)")

## What you built

Every tool call is now gated by a policy that fails closed, and every decision is an Ed25519 receipt anyone can verify offline. This is the building block for an auditable agent: who did what, under which policy, and whether it was allowed or denied, provable without trusting the operator.

- [protect-mcp](https://www.npmjs.com/package/protect-mcp) (the gate and signer) and [@veritasacta/verify](https://www.npmjs.com/package/@veritasacta/verify) (the offline verifier)
- Receipt wire format: [draft-farley-acta-signed-receipts](https://datatracker.ietf.org/doc/draft-farley-acta-signed-receipts/) (IETF Internet-Draft)